# Creative Ideas with Everyday Objects

Enter an everyday object and get three unusual, practical ideas from GPT-6 Luna. You can try as many objects as you like in one run.


## Before you run

Run the SDK installation cell first. Then run the idea generator. It asks for your OpenAI API key once using a hidden prompt, then keeps asking for objects. Type quit when you are done. Use an OpenAI API key, not your ChatGPT password. The key stays in this Colab runtime and is not written into the notebook.


In [ ]:
%pip -q install openai


In [ ]:
from getpass import getpass
from openai import (
    APIConnectionError,
    APIStatusError,
    AuthenticationError,
    OpenAI,
    RateLimitError,
)

# Ask for the API key once. getpass hides it while you type.
api_key = getpass("Enter your OpenAI API key (input is hidden): ")
if not api_key.strip():
    raise ValueError("Please enter your API key and run this cell again.")

client = OpenAI(api_key=api_key)
del api_key  # Do not print or save the key.

print("Enter an everyday object to get three ideas. Type quit to stop.")

while True:
    item = input("\nEveryday object (or quit): ").strip()

    if item.lower() == "quit":
        print("Finished. Thanks for trying the idea generator!")
        break

    if not item:
        print("Please enter an object, or type quit to stop.")
        continue

    try:
        response = client.responses.create(
            model="gpt-6-luna",
            instructions=(
                "Suggest exactly THREE unusual but practical uses for the everyday object. "
                "Return exactly three ideas, numbered 1, 2, and 3. Make them different, "
                "safe, legal, realistic, and low-cost. Give each idea a short title and "
                "one or two simple sentences. Do not add an introduction or extra ideas."
            ),
            input=f"Everyday object: {item}",
            reasoning={"effort": "low"},
            max_output_tokens=900,
        )

        # Show returned text, and explain empty or cut-off responses.
        ideas = (response.output_text or "").strip()
        status = getattr(response, "status", None) or "unknown"
        details = getattr(response, "incomplete_details", None)
        reason = getattr(details, "reason", None) if details else None

        if ideas:
            print(ideas)
            if status == "incomplete" and reason == "max_output_tokens":
                print("\nThe answer may be cut off because it reached the output limit.")
        elif status == "incomplete" and reason == "max_output_tokens":
            print("No ideas came back because the response reached its output limit. Please try again.")
        elif status != "completed":
            print(f"No ideas were returned (response status: {status}). Please check your connection and try again.")
        else:
            print("The request completed, but the API returned no text. Please try this object again.")

    except AuthenticationError:
        print("Authentication failed. Check that you entered a valid OpenAI API key, then rerun this cell.")
        break
    except RateLimitError:
        print("The API usage limit was reached. Check your OpenAI API usage and billing, or try again later.")
    except APIConnectionError:
        print("Could not connect to OpenAI. Check your internet connection, then try this object again.")
    except APIStatusError as error:
        print(f"OpenAI returned an HTTP {error.status_code} error. Check your API access, then try again.")
    except Exception as error:
        # Do not print raw errors, which could contain request or credential details.
        print(f"Something unexpected happened ({type(error).__name__}). Please try again.")
